Работа с табличными данными

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
from dotenv import load_dotenv
import numpy as np
import wandb
import os
import logging
import time
import torch.nn as nn
import random
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import *
import pickle

In [2]:
df = pd.read_csv('dataset/train.csv', sep="|")
df

,trustLevel,totalScanTimeInSeconds,grandTotal,lineItemVoids,scansWithoutRegistration,quantityModifications,scannedLineItemsPerSecond,valuePerSecond,lineItemVoidsPerPosition,fraud
0,5,1054,54.70,7,0,3,0.027514,0.051898,0.241379,0
1,3,108,27.36,5,2,4,0.129630,0.253333,0.357143,0
2,3,1516,62.16,3,10,5,0.008575,0.041003,0.230769,0
3,6,1791,92.31,8,4,4,0.016192,0.051541,0.275862,0
4,5,430,81.53,3,7,2,0.062791,0.189605,0.111111,0
...,...,...,...,...,...,...,...,...,...,...
1874,1,321,76.03,8,7,2,0.071651,0.236854,0.347826,0
1875,1,397,41.89,5,5,0,0.065491,0.105516,0.192308,1
1876,4,316,41.83,5,8,1,0.094937,0.132373,0.166667,0
1877,2,685,62.68,1,6,2,0.035036,0.091504,0.041667,0


In [3]:
df.dtypes

trustLevel                     int64
totalScanTimeInSeconds         int64
grandTotal                   float64
lineItemVoids                  int64
scansWithoutRegistration       int64
quantityModifications          int64
scannedLineItemsPerSecond    float64
valuePerSecond               float64
lineItemVoidsPerPosition     float64
fraud                          int64
dtype: object

In [4]:
df.isna().sum()

trustLevel                   0
totalScanTimeInSeconds       0
grandTotal                   0
lineItemVoids                0
scansWithoutRegistration     0
quantityModifications        0
scannedLineItemsPerSecond    0
valuePerSecond               0
lineItemVoidsPerPosition     0
fraud                        0
dtype: int64

In [5]:
df.fraud.value_counts()

fraud
0    1775
1     104
Name: count, dtype: int64

In [6]:
y = df["fraud"]
X = df.drop(columns=["fraud"])

In [7]:
df.describe()

,trustLevel,totalScanTimeInSeconds,grandTotal,lineItemVoids,scansWithoutRegistration,quantityModifications,scannedLineItemsPerSecond,valuePerSecond,lineItemVoidsPerPosition,fraud
count,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000
mean,3.401809,932.153273,50.864492,5.469931,4.904204,2.525279,0.058138,0.201746,0.745404,0.055349
std,1.709404,530.144640,28.940202,3.451169,3.139697,1.695472,0.278512,1.242135,1.327241,0.228720
min,1.000000,2.000000,0.010000,0.000000,0.000000,0.000000,0.000548,0.000007,0.000000,0.000000
25%,2.000000,474.500000,25.965000,2.000000,2.000000,1.000000,0.008384,0.027787,0.160000,0.000000
50%,3.000000,932.000000,51.210000,5.000000,5.000000,3.000000,0.016317,0.054498,0.350000,0.000000
75%,5.000000,1397.000000,77.285000,8.000000,8.000000,4.000000,0.032594,0.107313,0.666667,0.000000
max,6.000000,1831.000000,99.960000,11.000000,10.000000,5.000000,6.666667,37.870000,11.000000,1.000000


## Предобработка данных

In [8]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [9]:
X_train.shape

(1503, 9)

In [10]:
X_val.shape

(376, 9)

In [11]:
X_test = pd.read_csv("dataset/test.csv", sep="|")
X_test

,trustLevel,totalScanTimeInSeconds,grandTotal,lineItemVoids,scansWithoutRegistration,quantityModifications,scannedLineItemsPerSecond,valuePerSecond,lineItemVoidsPerPosition
0,4,467,88.48,4,8,4,0.014989,0.189465,0.571429
1,3,1004,58.99,7,6,1,0.026892,0.058755,0.259259
2,1,162,14.00,4,5,4,0.006173,0.086420,4.000000
3,5,532,84.79,9,3,4,0.026316,0.159380,0.642857
4,5,890,42.16,4,0,0,0.021348,0.047371,0.210526
...,...,...,...,...,...,...,...,...,...
498116,4,783,59.10,2,2,0,0.012771,0.075479,0.200000
498117,1,278,98.90,9,5,4,0.050360,0.355755,0.642857
498118,3,300,5.41,6,6,4,0.030000,0.018033,0.666667
498119,2,1524,33.97,2,5,3,0.005906,0.022290,0.222222


In [12]:
y_test = pd.read_csv("dataset/DMC-2019-realclass.csv", sep="|")["fraud"]
y_test.value_counts()

fraud
0    474394
1     23727
Name: count, dtype: int64

In [13]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [14]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

In [15]:
y_train = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_val = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

In [16]:
BATCH_SIZE = 64

In [17]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=256)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=1024)

вес для редкого класса, чтобы модель его не пропускала

In [18]:
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()

In [19]:
pos_weight = n_neg / n_pos
pos_weight

tensor(17.1084)

## Подготовка

In [20]:
load_dotenv()

WANDB_API_KEY = os.getenv("WANDB_API_KEY")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "gp5")
WANDB_ENTITY = os.getenv("WANDB_ENTITY")


In [21]:
wandb.login(key=WANDB_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/evgeniy/.netrc
wandb: Currently logged in as: gigantina-ru (gigantina-ru-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [22]:
def stop_logging():
    logger = logging.getLogger()
    for handler in logger.handlers:
        handler.flush()
        handler.close()
        logger.removeHandler(handler)

def new_log_file():
    stop_logging()
    timestamp = str(time.time()).replace('.', '_')
    log_file = f'part_2_{timestamp}.log'
    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        force=True
    )
    logging.info("Начал логгировать новый запуск")
    return log_file

In [23]:
def evaluate_model(model, X, y, threshold=0.5, name="dataset"):
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED) 

    model.eval()

    with torch.no_grad():
        logits = model(X)
        probs = torch.sigmoid(logits)

    y_true = y.numpy().ravel()
    y_prob = probs.numpy().ravel()
    y_pred = (y_prob > threshold).astype(int)

    roc_auc = roc_auc_score(y_true, y_prob)
    avg_precision = average_precision_score(y_true, y_prob)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f05 = fbeta_score(y_true, y_pred, beta=0.5)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    profit = tp * 5 - fp * 25 - fn * 5

    print(name)
    print("ROC-AUC:", round(roc_auc, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F0.5-score:", round(f05, 4))
    print("Profit:", profit)
    print()

    return {
        "dataset": name,
        "threshold": threshold,
        "roc_auc": roc_auc,
        "average_precision": avg_precision,
        "precision": precision,
        "recall": recall,
        "f05": f05,
        "profit": profit,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

In [24]:
import copy
import random

def train_model(
    model,
    train_loader,
    X_valid,
    y_valid,
    loss_fn,
    optimizer,
    epochs=30,
    threshold=0.5,
    model_name="model"
):
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED) 
    best_metric = -10**9
    best_epoch = 0
    best_state = None

    history = []

    logging.info(f"Начали обучение {model_name}")

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0

        for xb, yb in train_loader:
            optimizer.zero_grad()

            pred = model(xb)
            loss = loss_fn(pred, yb)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)

        valid_metrics = evaluate_model(
            model,
            X_valid,
            y_valid,
            threshold=threshold,
            name=f"{model_name} | valid epoch {epoch}"
        )

        current_metric = valid_metrics["roc_auc"]

        if current_metric > best_metric:
            best_metric = current_metric
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

        history.append({
            "epoch": epoch,
            "train_loss": epoch_loss,
            **valid_metrics
        })


        print(
            f"Эпоха {epoch:02d} | "
            f"Train Loss: {epoch_loss:.4f} | "
            f"Val ROC-AUC: {valid_metrics['roc_auc']:.4f} | "
            f"Val Profit: {valid_metrics['profit']}"
        )
        logging.info(
            f"Эпоха {epoch:02d} | "
            f"Train Loss: {epoch_loss:.4f} | "
            f"Val ROC-AUC: {valid_metrics['roc_auc']:.4f} | "
            f"Val Profit: {valid_metrics['profit']}"
        )

        wandb.log({
             f"{model_name}/train_loss": epoch_loss,
             f"{model_name}/valid_profit": valid_metrics["profit"],
             f"{model_name}/valid_roc_auc": valid_metrics["roc_auc"],
             f"{model_name}/valid_average_precision": valid_metrics["average_precision"],
             f"{model_name}/valid_precision": valid_metrics["precision"],
             f"{model_name}/valid_recall": valid_metrics["recall"],
             f"{model_name}/valid_f05": valid_metrics["f05"],
         })

    model.load_state_dict(best_state)

    print()
    print(f"Лучшая эпоха для {model_name}: {best_epoch}")
    print(f"Лучший ROC-AUC: {best_metric}")
    logging.info("Закончили обучение")
    logging.info(f"Лучшая эпоха для {model_name}: {best_epoch}")
    logging.info(f"Лучший ROC-AUC: {best_metric}")

    return model, history

In [25]:
def save_results(model, name, log_file, run):
    train_metrics = evaluate_model(model, X_train, y_train, threshold=0.5, name="Train")
    test_metrics = evaluate_model(model, X_test, y_test, threshold=0.5, name="Test")
    pickle.dump(model.state_dict(), open(f"models/{name}.pkl", 'wb'))
    logging.info("Сохранили веса модели в папку models")

    metric_keys = list(train_metrics.keys())

    table = wandb.Table(columns=metric_keys)
    table.add_data(*[train_metrics[i] for i in metric_keys])
    table.add_data(*[test_metrics[i] for i in metric_keys])
    wandb.log({'results': table})
    artifact = wandb.Artifact(name=name, type="model", description=f"Тест логгирования модели: {name}")

    artifact.add_file(f"models/{name}.pkl")
    artifact.add_file(log_file)
    run.log_artifact(artifact)

## model_1_baseline

In [26]:
EPOCHS = 30
LR = 0.01
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 

model_1 = nn.Sequential(
    nn.Linear(9, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_1.parameters(), lr=LR)

config = {
    "model": "MLP",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_1)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_1_baseline", config=config)
log_file = new_log_file()

model_1, history_1 = train_model(
    model=model_1,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_1_baseline"
)

save_results(model_1, "model_1_baseline", log_file, run)


run.finish()

model_1_baseline | valid epoch 1
ROC-AUC: 0.9411
Precision: 0.25
Recall: 0.0952
F0.5-score: 0.1887
Profit: -235

Эпоха 01 | Train Loss: 1.2482 | Val ROC-AUC: 0.9411 | Val Profit: -235
model_1_baseline | valid epoch 2
ROC-AUC: 0.9553
Precision: 0.2188
Recall: 1.0
F0.5-score: 0.2593
Profit: -1770

Эпоха 02 | Train Loss: 0.7541 | Val ROC-AUC: 0.9553 | Val Profit: -1770
model_1_baseline | valid epoch 3
ROC-AUC: 0.972
Precision: 0.2561
Recall: 1.0
F0.5-score: 0.3009
Profit: -1420

Эпоха 03 | Train Loss: 0.4640 | Val ROC-AUC: 0.9720 | Val Profit: -1420
model_1_baseline | valid epoch 4
ROC-AUC: 0.9728
Precision: 0.2625
Recall: 1.0
F0.5-score: 0.3079
Profit: -1370

Эпоха 04 | Train Loss: 0.3839 | Val ROC-AUC: 0.9728 | Val Profit: -1370
model_1_baseline | valid epoch 5
ROC-AUC: 0.972
Precision: 0.2763
Recall: 1.0
F0.5-score: 0.3231
Profit: -1270

Эпоха 05 | Train Loss: 0.3468 | Val ROC-AUC: 0.9720 | Val Profit: -1270
model_1_baseline | valid epoch 6
ROC-AUC: 0.9741
Precision: 0.2838
Recall: 1.0

model_1_baseline/train_loss,█▅▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_1_baseline/valid_average_precision,▁▂▅▅▅▅▅▅▅▅▅▅▅▅▆▆▆▆▇▆▆▇▆▇▆▆▅▅▅█
model_1_baseline/valid_f05,▁▂▃▃▃▃▄▄▄▅▆▅▆▆▇▆▇▇▇▇▇█▇▆██▆▆█▆
model_1_baseline/valid_precision,▂▁▂▂▂▂▃▃▃▄▅▅▅▅▇▅▆▆▆▇▇█▆▆█▇▆▅█▆
model_1_baseline/valid_profit,█▁▃▃▃▄▄▅▅▅▆▆▆▆▇▆▇▇▇▇▇█▇▆▇▇▇▆█▆
model_1_baseline/valid_recall,▁██████████████████▇██████▇█▇█
model_1_baseline/valid_roc_auc,▁▃▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇█▇▇█▇███▆▆▇█
model_1_baseline/train_loss,0.12749
model_1_baseline/valid_average_precision,0.84219
model_1_baseline/valid_f05,0.49296
model_1_baseline/valid_precision,0.4375


Видно сильное переобучение! Я хз, что с этим делать

### Попробуем подобрать порог

In [ ]:
with torch.no_grad():
    val_prob = torch.sigmoid(model(X_val)).numpy()

val_true = y_val.numpy()

In [ ]:
from sklearn.metrics import precision_recall_curve

target_recall = 0.90

prec, rec, thr = precision_recall_curve(val_true, val_prob)
best_threshold = thr[rec[:-1] >= target_recall].max()
logging.info(f"Подобрали порог {best_threshold}")
wandb.log({"best_threshold": best_threshold})
best_threshold

In [ ]:
y_pred = (y_prob > best_threshold).astype(int)

In [ ]:
pre_at_score = precision_score(y_true, y_pred)
#logging.info(f"На тесте Precision: {pre_at_score}")
pre_at_score

In [ ]:
rec_at_score = recall_score(y_true, y_pred)
#logging.info(f"На тесте Recall: {rec_at_score}")
rec_at_score

In [ ]:
import matplotlib.pyplot as plt

plt.plot(rec, prec)
plt.xlabel("recall")
plt.ylabel("precision")
plt.title("PR кривая")
plt.show()

In [ ]:
#import pickle

#pickle.dump(model.state_dict(), open("models/model_fraud_1.pkl", 'wb'))
#logging.info("Сохранили веса модели в папку models")

In [ ]:
#results = wandb.Table(columns=['model', 'test/roc_auc', 'test/precision', 'test/recall', 'test/recall_new_threshold', 'test/precision_new_threshold'])
#results.add_data("nn_baseline", auc_score, pre_score, rec_score, rec_at_score, pre_at_score)
#wandb.log({"results": results})

In [ ]:
#artifact = wandb.Artifact(name="model_fraud_1", type="model", description="Тест логгирования модели MLP")

#artifact.add_file("models/model_fraud_1.pkl")
#artifact.add_file(log_file)
#run.log_artifact(artifact)




In [ ]:
#run.finish()

## model_2_dop_sloi

In [ ]:
EPOCHS = 30
LR = 0.01

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_2 = nn.Sequential(
    nn.Linear(9, 32),
    nn.ReLU(),

    nn.Linear(32, 16),
    nn.ReLU(),

    nn.Linear(16, 8),
    nn.ReLU(),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_2.parameters(), lr=LR)

config = {
    "model": "MLP_added_layer",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_2)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_2_added_layers", config=config)
log_file = new_log_file()

model_2, history_2 = train_model(
    model=model_2,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_2_dop_sloi"
)

save_results(model_2, "model_2", log_file, run)

run.finish()

model_2_dop_sloi | valid epoch 1
ROC-AUC: 0.9311
Precision: 0.3051
Recall: 0.8571
F0.5-score: 0.3502
Profit: -950

Эпоха 01 | Train Loss: 1.1825 | Val ROC-AUC: 0.9311 | Val Profit: -950
model_2_dop_sloi | valid epoch 2
ROC-AUC: 0.9552
Precision: 0.3
Recall: 1.0
F0.5-score: 0.3488
Profit: -1120

Эпоха 02 | Train Loss: 0.6270 | Val ROC-AUC: 0.9552 | Val Profit: -1120
model_2_dop_sloi | valid epoch 3
ROC-AUC: 0.9502
Precision: 0.25
Recall: 1.0
F0.5-score: 0.2941
Profit: -1470

Эпоха 03 | Train Loss: 0.4225 | Val ROC-AUC: 0.9502 | Val Profit: -1470
model_2_dop_sloi | valid epoch 4
ROC-AUC: 0.9642
Precision: 0.2857
Recall: 0.9524
F0.5-score: 0.3322
Profit: -1155

Эпоха 04 | Train Loss: 0.4105 | Val ROC-AUC: 0.9642 | Val Profit: -1155
model_2_dop_sloi | valid epoch 5
ROC-AUC: 0.9641
Precision: 0.3182
Recall: 1.0
F0.5-score: 0.3684
Profit: -1020

Эпоха 05 | Train Loss: 0.3026 | Val ROC-AUC: 0.9641 | Val Profit: -1020
model_2_dop_sloi | valid epoch 6
ROC-AUC: 0.9599
Precision: 0.2941
Recall: 0

model_2_dop_sloi/train_loss,█▄▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▂▁▁▃▂▁▁▁▁▁▂▂
model_2_dop_sloi/valid_average_precision,▁▃▃▄▃▃▃▃▃▄▄▅▆▅▄▅▅▆▅▅▅▄▅▇▆█▇▇▅▆
model_2_dop_sloi/valid_f05,▂▂▁▂▂▂▃▃▄▃▄▄▅▅▅▆▆▇▇▄▆▂▄▅▇█▇█▂▄
model_2_dop_sloi/valid_precision,▂▂▁▂▂▂▃▃▄▃▃▃▄▄▄▆▆▇▇▄▆▂▄▅▆█▆█▂▄
model_2_dop_sloi/valid_profit,▄▃▁▃▃▃▄▅▆▅▅▅▆▆▆▇▇▇█▅▇▃▆▆▇█▇█▃▅
model_2_dop_sloi/valid_recall,▄██▇█▇▇▇▁█▅█▇▄▅▅▅▅▄█▂███▅▂▇▄██
model_2_dop_sloi/valid_roc_auc,▁▄▃▅▅▅▅▅▅▅▅▆▇▆▆▇▇▇▇▆▆▅▆▇███▇▆▇
model_2_dop_sloi/train_loss,0.31125
model_2_dop_sloi/valid_average_precision,0.71646
model_2_dop_sloi/valid_f05,0.46667
model_2_dop_sloi/valid_precision,0.41176


In [ ]:
train_metrics = evaluate_model(model_2, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_2, X_test, y_test, threshold=0.5, name="Test")

## model_3_bolshe_neyronov

In [28]:
EPOCHS = 30
LR = 0.01

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 

model_3 = nn.Sequential(
    nn.Linear(9, 32),
    nn.ReLU(),
    nn.Linear(32, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_3.parameters(), lr=LR)

config = {
    "model": "MLP_added_neurons",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_3)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_3_more_neurons", config=config)
log_file = new_log_file()

model_3, history_3 = train_model(
    model=model_3,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_3_bolshe_neyronov"
)


save_results(model_3, "model_3", log_file, run)

run.finish()



model_3_bolshe_neyronov | valid epoch 1
ROC-AUC: 0.9586
Precision: 0.2442
Recall: 1.0
F0.5-score: 0.2877
Profit: -1520

Эпоха 01 | Train Loss: 0.9062 | Val ROC-AUC: 0.9586 | Val Profit: -1520
model_3_bolshe_neyronov | valid epoch 2
ROC-AUC: 0.9599
Precision: 0.3043
Recall: 1.0
F0.5-score: 0.3535
Profit: -1095

Эпоха 02 | Train Loss: 0.4444 | Val ROC-AUC: 0.9599 | Val Profit: -1095
model_3_bolshe_neyronov | valid epoch 3
ROC-AUC: 0.9612
Precision: 0.3279
Recall: 0.9524
F0.5-score: 0.3774
Profit: -930

Эпоха 03 | Train Loss: 0.3691 | Val ROC-AUC: 0.9612 | Val Profit: -930
model_3_bolshe_neyronov | valid epoch 4
ROC-AUC: 0.9679
Precision: 0.3922
Recall: 0.9524
F0.5-score: 0.4444
Profit: -680

Эпоха 04 | Train Loss: 0.3041 | Val ROC-AUC: 0.9679 | Val Profit: -680
model_3_bolshe_neyronov | valid epoch 5
ROC-AUC: 0.9658
Precision: 0.3704
Recall: 0.9524
F0.5-score: 0.4219
Profit: -755

Эпоха 05 | Train Loss: 0.2697 | Val ROC-AUC: 0.9658 | Val Profit: -755
model_3_bolshe_neyronov | valid epoch

IOStream.flush timed out


model_3_bolshe_neyronov/train_loss,█▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▂▃▂▁▁▁▁
model_3_bolshe_neyronov/valid_average_precision,▃▃▁▂▁▂▃▄▂▃▃▁▂▄▅▅▆▄▄▅▄▅▆▆▆▆▇█▆▆
model_3_bolshe_neyronov/valid_f05,▁▂▃▄▃▃▅▄▅▄▄▅▄▅▅▅▆▆▇▇▆▇█▄▄▇▇█▇█
model_3_bolshe_neyronov/valid_precision,▁▂▃▄▃▃▅▄▄▄▄▅▄▅▅▅▆▆▇▇▆▇█▄▄▆▇█▆█
model_3_bolshe_neyronov/valid_profit,▁▃▄▅▅▄▇▆▆▅▅▆▅▆▇▆▇▇██▇██▆▆▇██▇█
model_3_bolshe_neyronov/valid_recall,██▇▇▇▇▄▇▅█▇▅█▅▇▄▄▇▇▅▇▁▇▇▇▇▇▇▇▇
model_3_bolshe_neyronov/valid_roc_auc,▁▁▂▃▃▄▄▅▄▄▅▃▄▃▅▆▆▆▆▆▆▆▇▅▅▇▇█▇▇
model_3_bolshe_neyronov/train_loss,0.07161
model_3_bolshe_neyronov/valid_average_precision,0.6521
model_3_bolshe_neyronov/valid_f05,0.67114
model_3_bolshe_neyronov/valid_precision,0.625


In [ ]:
train_metrics = evaluate_model(model_3, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_3, X_test, y_test, threshold=0.5, name="Test")

## model_4_tolko_batchnorm

In [ ]:
EPOCHS = 30
LR = 0.01

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_4 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),

    nn.Linear(8, 1)
)



run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_4_batchnorm_only", config=config)
log_file = new_log_file()

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_4.parameters(), lr=LR)

config = {
    "model": "MLP_batchnorm_only",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_4)
}

model_4, history_4 = train_model(
    model=model_4,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_4_tolko_batchnorm"
)


save_results(model_4, "model_4", log_file, run)

run.finish()

model_4_tolko_batchnorm | valid epoch 1
ROC-AUC: 0.9535
Precision: 0.1355
Recall: 1.0
F0.5-score: 0.1638
Profit: -3245

Эпоха 01 | Train Loss: 0.9802 | Val ROC-AUC: 0.9535 | Val Profit: -3245
model_4_tolko_batchnorm | valid epoch 2
ROC-AUC: 0.9645
Precision: 0.21
Recall: 1.0
F0.5-score: 0.2494
Profit: -1870

Эпоха 02 | Train Loss: 0.5630 | Val ROC-AUC: 0.9645 | Val Profit: -1870
model_4_tolko_batchnorm | valid epoch 3
ROC-AUC: 0.9599
Precision: 0.25
Recall: 1.0
F0.5-score: 0.2941
Profit: -1470

Эпоха 03 | Train Loss: 0.4020 | Val ROC-AUC: 0.9599 | Val Profit: -1470
model_4_tolko_batchnorm | valid epoch 4
ROC-AUC: 0.9634
Precision: 0.25
Recall: 1.0
F0.5-score: 0.2941
Profit: -1470

Эпоха 04 | Train Loss: 0.3919 | Val ROC-AUC: 0.9634 | Val Profit: -1470
model_4_tolko_batchnorm | valid epoch 5
ROC-AUC: 0.9767
Precision: 0.3559
Recall: 1.0
F0.5-score: 0.4086
Profit: -845

Эпоха 05 | Train Loss: 0.3272 | Val ROC-AUC: 0.9767 | Val Profit: -845
model_4_tolko_batchnorm | valid epoch 6
ROC-AUC:

model_4_tolko_batchnorm/train_loss,█▅▃▃▃▂▃▃▂▂▁▃▃▂▂▂▁▁▂▄▃▂▂▂▂▁▁▁▁▂
model_4_tolko_batchnorm/valid_average_precision,▂▃▃▃▆▄▅█▇▇▁▅▄▄▅▄▅▇▄▃▅▆▅▅▄▅▅▅▅▃
model_4_tolko_batchnorm/valid_f05,▁▂▃▃▄▅▃▅█▇▂▂▄▆▇▆▆▇▃▂▄▅▆▇▇▇▃▃▂▃
model_4_tolko_batchnorm/valid_precision,▁▂▂▂▄▅▃▅██▃▂▄▅▆▆▇▇▃▂▄▅▆▇▇▇▄▅▅▃
model_4_tolko_batchnorm/valid_profit,▁▄▅▅▆▇▅▇███▄▆▇████▆▅▇▇███████▆
model_4_tolko_batchnorm/valid_recall,█████▆██▆▄▁██▇▇▅▄▆████▇▆▅▅▁▁▁█
model_4_tolko_batchnorm/valid_roc_auc,▇▇▇▇█▇▇███▁███████▇▇█████████▇
model_4_tolko_batchnorm/train_loss,0.17732
model_4_tolko_batchnorm/valid_average_precision,0.50095
model_4_tolko_batchnorm/valid_f05,0.35587
model_4_tolko_batchnorm/valid_precision,0.30769


In [ ]:
train_metrics = evaluate_model(model_4, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_4, X_test, y_test, threshold=0.5, name="Test")

Батч норм вообще не помог, а только ухудшил качество

## model_5_tolko_dropout

In [30]:
EPOCHS = 40
LR = 0.01
DROPOUT_COEF = 0.3

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_5 = nn.Sequential(
    nn.Linear(9, 128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_5.parameters(), lr=LR)

config = {
    "model": "MLP_dropout_only",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "dropout_coef": DROPOUT_COEF,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_5)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_5_dropout_only", config=config)
log_file = new_log_file()

model_5, history_5 = train_model(
    model=model_5,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_5_tolko_dropout"
)

save_results(model_5, "model_5", log_file, run)

run.finish()


model_5_tolko_dropout | valid epoch 1
ROC-AUC: 0.9489
Precision: 0.3929
Recall: 0.5238
F0.5-score: 0.4135
Profit: -420

Эпоха 01 | Train Loss: 1.2416 | Val ROC-AUC: 0.9489 | Val Profit: -420
model_5_tolko_dropout | valid epoch 2
ROC-AUC: 0.8999
Precision: 0.1927
Recall: 1.0
F0.5-score: 0.2298
Profit: -2095

Эпоха 02 | Train Loss: 0.8616 | Val ROC-AUC: 0.8999 | Val Profit: -2095
model_5_tolko_dropout | valid epoch 3
ROC-AUC: 0.8909
Precision: 0.2165
Recall: 1.0
F0.5-score: 0.2567
Profit: -1795

Эпоха 03 | Train Loss: 0.6937 | Val ROC-AUC: 0.8909 | Val Profit: -1795
model_5_tolko_dropout | valid epoch 4
ROC-AUC: 0.9411
Precision: 0.2211
Recall: 1.0
F0.5-score: 0.2618
Profit: -1745

Эпоха 04 | Train Loss: 0.5990 | Val ROC-AUC: 0.9411 | Val Profit: -1745
model_5_tolko_dropout | valid epoch 5
ROC-AUC: 0.9505
Precision: 0.3175
Recall: 0.9524
F0.5-score: 0.3663
Profit: -980

Эпоха 05 | Train Loss: 0.5082 | Val ROC-AUC: 0.9505 | Val Profit: -980
model_5_tolko_dropout | valid epoch 6
ROC-AUC: 0

model_5_tolko_dropout/train_loss,█▆▅▄▄▄▃▂▂▃▃▃▂▂▂▂▂▂▂▂▂▂▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂
model_5_tolko_dropout/valid_average_precision,▅▃▁▄▅▅▅▅▅▇▇▇▇▆▇▇▅█████▅▆▄▆▇▇▇██▇▇▇▆▆▆▆▆▆
model_5_tolko_dropout/valid_f05,▅▁▂▂▄▄▆▅▄▅▅▄▅▅▅▆▄▆▆▇▇█▃▄▅▅▆▆▇▇█▇█████▆▇▆
model_5_tolko_dropout/valid_precision,▅▁▁▂▃▄▅▄▃▄▄▃▄▅▅▆▄▅▅▇▆▇▃▃▄▅▅▆▇▇▇▇▇█▇▇█▆▇▅
model_5_tolko_dropout/valid_profit,▇▁▂▂▅▆▇▆▅▆▆▅▆▆▇▇▇▆▆▇▇▇▅▅▆▇▇▇█████████▇█▇
model_5_tolko_dropout/valid_recall,▁███▇▇▆▆▇███▇▇▅▄▃▇▇▅▄▅▅▇▅▄▄▂▂▁▁▁▁▁▂▁▁▂▁▂
model_5_tolko_dropout/valid_roc_auc,▆▂▁▅▆▆▇▇▆▇▇█▇▇▇▇▆█████▅▆▅▆▇▇▇▇█▇▇▇▇▇▇▇▆▇
model_5_tolko_dropout/train_loss,0.1764
model_5_tolko_dropout/valid_average_precision,0.52402
model_5_tolko_dropout/valid_f05,0.46099
model_5_tolko_dropout/valid_precision,0.43333


In [ ]:
train_metrics = evaluate_model(model_5, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_5, X_test, y_test, threshold=0.5, name="Test")

## model_6_batchnorm_i_dropout

In [31]:
EPOCHS = 30
LR = 0.01
DROPOUT_COEF = 0.3
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_6 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_6.parameters(), lr=LR)

config = {
    "model": "MLP_batchnorm_dropout",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "dropout_coef": DROPOUT_COEF,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_6)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_6_batchnorm_dropout", config=config)
log_file = new_log_file()

model_6, history_6 = train_model(
    model=model_6,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_6_batchnorm_i_dropout"
)


save_results(model_6, "model_6", log_file, run)

run.finish()

model_6_batchnorm_i_dropout | valid epoch 1
ROC-AUC: 0.9081
Precision: 0.1273
Recall: 1.0
F0.5-score: 0.1542
Profit: -3495

Эпоха 01 | Train Loss: 1.2793 | Val ROC-AUC: 0.9081 | Val Profit: -3495
model_6_batchnorm_i_dropout | valid epoch 2
ROC-AUC: 0.9441
Precision: 0.1257
Recall: 1.0
F0.5-score: 0.1524
Profit: -3545

Эпоха 02 | Train Loss: 1.0165 | Val ROC-AUC: 0.9441 | Val Profit: -3545
model_6_batchnorm_i_dropout | valid epoch 3
ROC-AUC: 0.9591
Precision: 0.2234
Recall: 1.0
F0.5-score: 0.2645
Profit: -1720

Эпоха 03 | Train Loss: 0.6499 | Val ROC-AUC: 0.9591 | Val Profit: -1720
model_6_batchnorm_i_dropout | valid epoch 4
ROC-AUC: 0.9584
Precision: 0.475
Recall: 0.9048
F0.5-score: 0.5249
Profit: -440

Эпоха 04 | Train Loss: 0.3784 | Val ROC-AUC: 0.9584 | Val Profit: -440
model_6_batchnorm_i_dropout | valid epoch 5
ROC-AUC: 0.9333
Precision: 0.3684
Recall: 0.6667
F0.5-score: 0.4046
Profit: -565

Эпоха 05 | Train Loss: 0.2500 | Val ROC-AUC: 0.9333 | Val Profit: -565
model_6_batchnorm_i

model_6_batchnorm_i_dropout/train_loss,█▇▅▃▂▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_6_batchnorm_i_dropout/valid_average_precision,▁▅▆█▄▃▄▆▃▃▄▃▅▅▅▅▅▅▅▅▆▅▆▆▆▇█▇▇▇
model_6_batchnorm_i_dropout/valid_f05,▁▁▃█▆▄▄▆▆▄▅▅▆▆▆▆▆▆▆▆▆▆▆▆▅▆▆▄▄▄
model_6_batchnorm_i_dropout/valid_precision,▁▁▃▆▅▃▃▅▅▄▅▆▆▆▆▆▆▇▇▇▇▇▇▇▆▇█▆▆▇
model_6_batchnorm_i_dropout/valid_profit,▁▁▅▇▇▆▆▇█▇████████████████████
model_6_batchnorm_i_dropout/valid_recall,███▇▅▇██▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁
model_6_batchnorm_i_dropout/valid_roc_auc,▁▅▇▇▄▃▆█▆▃▃▂▃▃▃▃▃▃▃▃▃▃▃▃▃▃▄▃▃▃
model_6_batchnorm_i_dropout/train_loss,0.00448
model_6_batchnorm_i_dropout/valid_average_precision,0.51916
model_6_batchnorm_i_dropout/valid_f05,0.33333
model_6_batchnorm_i_dropout/valid_precision,0.5


In [ ]:
train_metrics = evaluate_model(model_6, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_6, X_test, y_test, threshold=0.5, name="Test")

## model_7_leaky_relu

In [32]:
EPOCHS = 30
LR = 0.01
DROPOUT_COEF = 0.3
NEG_SLOPE = 0.02

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_7 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_7.parameters(), lr=LR)

config = {
    "model": "MLP_leaky_relu",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "dropout_coef": DROPOUT_COEF,
    "negative_slope": NEG_SLOPE,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_7)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_7_leaky_relu", config=config)
log_file = new_log_file()

model_7, history_7 = train_model(
    model=model_7,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_7_leaky_relu"
)
save_results(model_7, "model_7", log_file, run)

run.finish()

model_7_leaky_relu | valid epoch 1
ROC-AUC: 0.8566
Precision: 0.1468
Recall: 0.7619
F0.5-score: 0.1751
Profit: -2270

Эпоха 01 | Train Loss: 1.2706 | Val ROC-AUC: 0.8566 | Val Profit: -2270
model_7_leaky_relu | valid epoch 2
ROC-AUC: 0.9344
Precision: 0.1228
Recall: 1.0
F0.5-score: 0.1489
Profit: -3645

Эпоха 02 | Train Loss: 0.9943 | Val ROC-AUC: 0.9344 | Val Profit: -3645
model_7_leaky_relu | valid epoch 3
ROC-AUC: 0.9513
Precision: 0.236
Recall: 1.0
F0.5-score: 0.2785
Profit: -1595

Эпоха 03 | Train Loss: 0.6864 | Val ROC-AUC: 0.9513 | Val Profit: -1595
model_7_leaky_relu | valid epoch 4
ROC-AUC: 0.9231
Precision: 0.5789
Recall: 0.5238
F0.5-score: 0.567
Profit: -195

Эпоха 04 | Train Loss: 0.4064 | Val ROC-AUC: 0.9231 | Val Profit: -195
model_7_leaky_relu | valid epoch 5
ROC-AUC: 0.8974
Precision: 0.4286
Recall: 0.4286
F0.5-score: 0.4286
Profit: -315

Эпоха 05 | Train Loss: 0.2869 | Val ROC-AUC: 0.8974 | Val Profit: -315
model_7_leaky_relu | valid epoch 6
ROC-AUC: 0.9264
Precision: 

model_7_leaky_relu/train_loss,█▆▅▃▃▃▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_7_leaky_relu/valid_average_precision,▁▄▆▅▃▇█▆▆▅▅▄▄▃▃▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄
model_7_leaky_relu/valid_f05,▃▃▄█▆▇▅▅▇▆▅▄▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▁▁
model_7_leaky_relu/valid_precision,▃▂▄█▆▆▄▅▇▆▆▆▄▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▁▁
model_7_leaky_relu/valid_profit,▄▁▅██▇▆▆██████████████████████
model_7_leaky_relu/valid_recall,▆██▅▄▇██▆▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_7_leaky_relu/valid_roc_auc,▂▆▇▆▄▆███▇▆▄▄▃▃▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂
model_7_leaky_relu/train_loss,0.00441
model_7_leaky_relu/valid_average_precision,0.38926
model_7_leaky_relu/valid_f05,0
model_7_leaky_relu/valid_precision,0


In [ ]:
train_metrics = evaluate_model(model_7, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_7, X_test, y_test, threshold=0.5, name="Test")

## model_8_weight_decay

In [33]:
EPOCHS = 30
LR = 0.01
WEIGHT_DECAY=1e-4
DROPOUT_COEF = 0.3

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_8 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_8.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

config = {
    "model": "MLP_weight_decay",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_8)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_8_weight_decay", config=config)
log_file = new_log_file()

model_8, history_8 = train_model(
    model=model_8,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_8_weight_decay"
)

save_results(model_8, "model_8", log_file, run)

run.finish()

model_8_weight_decay | valid epoch 1
ROC-AUC: 0.8787
Precision: 0.1313
Recall: 1.0
F0.5-score: 0.1589
Profit: -3370

Эпоха 01 | Train Loss: 1.2815 | Val ROC-AUC: 0.8787 | Val Profit: -3370
model_8_weight_decay | valid epoch 2
ROC-AUC: 0.9628
Precision: 0.1257
Recall: 1.0
F0.5-score: 0.1524
Profit: -3545

Эпоха 02 | Train Loss: 1.0369 | Val ROC-AUC: 0.9628 | Val Profit: -3545
model_8_weight_decay | valid epoch 3
ROC-AUC: 0.9412
Precision: 0.2041
Recall: 0.9524
F0.5-score: 0.2421
Profit: -1855

Эпоха 03 | Train Loss: 0.6819 | Val ROC-AUC: 0.9412 | Val Profit: -1855
model_8_weight_decay | valid epoch 4
ROC-AUC: 0.9371
Precision: 0.3061
Recall: 0.7143
F0.5-score: 0.3456
Profit: -805

Эпоха 04 | Train Loss: 0.4109 | Val ROC-AUC: 0.9371 | Val Profit: -805
model_8_weight_decay | valid epoch 5
ROC-AUC: 0.9319
Precision: 0.3182
Recall: 0.3333
F0.5-score: 0.3211
Profit: -410

Эпоха 05 | Train Loss: 0.2375 | Val ROC-AUC: 0.9319 | Val Profit: -410
model_8_weight_decay | valid epoch 6
ROC-AUC: 0.93

model_8_weight_decay/train_loss,█▇▅▃▂▂▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_8_weight_decay/valid_average_precision,▁▇▆▅▃▄▆▇██▇▆▆▆▇▇▆▅▆▆▆▆▆▆▇▇▆▆▆▇
model_8_weight_decay/valid_f05,▃▃▄▅▅▇▅▆▆█▇▇▆▆▆▄▃▃▃▃▃▁▁▁▁▁▁▁▁▁
model_8_weight_decay/valid_precision,▂▂▃▄▅▆▄▅▅▇█████▇▅▄▅▅▅▁▁▁▁▁▁▁▁▁
model_8_weight_decay/valid_profit,▁▁▄▇▇█▆▇▇█████████████████████
model_8_weight_decay/valid_recall,███▆▃▅▇▇▇▅▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_8_weight_decay/valid_roc_auc,▁█▆▆▅▅▅▇██████████████████████
model_8_weight_decay/train_loss,0.00478
model_8_weight_decay/valid_average_precision,0.49603
model_8_weight_decay/valid_f05,0
model_8_weight_decay/valid_precision,0


In [ ]:
train_metrics = evaluate_model(model_8, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_8, X_test, y_test, threshold=0.5, name="Test")

При этом пришлось обратно вернуться к обычной RELU

## model_9_Focal_loss

In [34]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        targets = targets.float()

        bce_loss = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none"
        )

        probs = torch.sigmoid(logits)

        pt = torch.where(targets == 1, probs, 1 - probs)

        focal_weight = self.alpha * (1 - pt) ** self.gamma

        loss = focal_weight * bce_loss

        return loss.mean()

In [35]:
EPOCHS = 30
LR = 0.01
DROPOUT_COEF = 0.3
SEED = 42
WEIGHT_DECAY = 1e-4
ALPHA = 0.25
GAMMA = 2.0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_9 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = FocalLoss(alpha=ALPHA, gamma=GAMMA)
optimizer = torch.optim.Adam(model_9.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

config = {
    "model": "MLP_focal_loss",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "alpha": ALPHA,
    "gamma": GAMMA,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_8)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_9_focal_loss", config=config)
log_file = new_log_file()

model_9, history_9 = train_model(
    model=model_9,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_9_Focal_loss"
)


save_results(model_9, "model_9", log_file, run)

run.finish()

/Users/evgeniy/cnn_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/evgeniy/cnn_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/evgeniy/cnn_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_9_Focal_loss | valid epoch 1
ROC-AUC: 0.8936
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 01 | Train Loss: 0.0233 | Val ROC-AUC: 0.8936 | Val Profit: -105
model_9_Focal_loss | valid epoch 2
ROC-AUC: 0.9335
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 02 | Train Loss: 0.0136 | Val ROC-AUC: 0.9335 | Val Profit: -105
model_9_Focal_loss | valid epoch 3
ROC-AUC: 0.9434
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 03 | Train Loss: 0.0097 | Val ROC-AUC: 0.9434 | Val Profit: -105


/Users/evgeniy/cnn_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/evgeniy/cnn_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/evgeniy/cnn_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_9_Focal_loss | valid epoch 4
ROC-AUC: 0.958
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 04 | Train Loss: 0.0080 | Val ROC-AUC: 0.9580 | Val Profit: -105
model_9_Focal_loss | valid epoch 5
ROC-AUC: 0.9663
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 05 | Train Loss: 0.0074 | Val ROC-AUC: 0.9663 | Val Profit: -105
model_9_Focal_loss | valid epoch 6
ROC-AUC: 0.9531
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 06 | Train Loss: 0.0064 | Val ROC-AUC: 0.9531 | Val Profit: -105
model_9_Focal_loss | valid epoch 7
ROC-AUC: 0.9375
Precision: 0.5
Recall: 0.2381
F0.5-score: 0.4098
Profit: -180

Эпоха 07 | Train Loss: 0.0051 | Val ROC-AUC: 0.9375 | Val Profit: -180
model_9_Focal_loss | valid epoch 8
ROC-AUC: 0.9524
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 08 | Train Loss: 0.0054 | Val ROC-AUC: 0.9524 | Val Profit: -105
model_9_Focal_loss | valid epoch 9
ROC-AUC: 0.9457
Precision: 1.0
Recall: 0.0476
F0.5-score: 0.2
Pro

/Users/evgeniy/cnn_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/evgeniy/cnn_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


...

/Users/evgeniy/cnn_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_9_Focal_loss | valid epoch 28
ROC-AUC: 0.9687
Precision: 0.5
Recall: 0.0952
F0.5-score: 0.2703
Profit: -135

Эпоха 28 | Train Loss: 0.0014 | Val ROC-AUC: 0.9687 | Val Profit: -135
model_9_Focal_loss | valid epoch 29
ROC-AUC: 0.9069
Precision: 1.0
Recall: 0.0476
F0.5-score: 0.2
Profit: -95

Эпоха 29 | Train Loss: 0.0013 | Val ROC-AUC: 0.9069 | Val Profit: -95
model_9_Focal_loss | valid epoch 30
ROC-AUC: 0.8846
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 30 | Train Loss: 0.0012 | Val ROC-AUC: 0.8846 | Val Profit: -105

Лучшая эпоха для model_9_Focal_loss: 11
Лучший ROC-AUC: 0.9722334004024145
Train
ROC-AUC: 0.9651
Precision: 1.0
Recall: 0.0964
F0.5-score: 0.3478
Profit: -335



/Users/evgeniy/cnn_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Test
ROC-AUC: 0.9568
Precision: 0.8461
Recall: 0.0943
F0.5-score: 0.3262
Profit: -106430



model_9_Focal_loss/train_loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▂▂▁▁▁▁▁▁▁
model_9_Focal_loss/valid_average_precision,▁▃▃▇█▆▅█▆▆█▆▆▇▇▅▇▆▆▄▅▂▆▅▆▅▆▇▄▆
model_9_Focal_loss/valid_f05,▁▁▁▁▁▁▇▁▄▁▇██▄▄▄▇▃▇▄▆▅▆▁▄▄▁▅▄▁
model_9_Focal_loss/valid_precision,▁▁▁▁▁▁▅▁█▁█▄▇████▃█▄▅▃▅▁██▁▅█▁
model_9_Focal_loss/valid_profit,▇▇▇▇▇▇▅▇▇▇█▁█▇▇▇█▆█▅▇▄▇▇▇▇▇▆▇▇
model_9_Focal_loss/valid_recall,▁▁▁▁▁▁▄▁▂▁▃█▃▂▂▂▃▂▃▂▃▃▃▁▂▂▁▂▂▁
model_9_Focal_loss/valid_roc_auc,▂▅▆▇█▆▅▆▆▆█▆▆▇▇▆▇▇▅▄▆▄▆▂█▄▅█▃▁
model_9_Focal_loss/train_loss,0.00121
model_9_Focal_loss/valid_average_precision,0.54825
model_9_Focal_loss/valid_f05,0
model_9_Focal_loss/valid_precision,0


In [ ]:
train_metrics = evaluate_model(model_9, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_9, X_test, y_test, threshold=0.5, name="Test")

# ансамбль ПОКА НЕ ТРОГАЛ

In [ ]:
import numpy as np

In [ ]:
def build_model():
    return nn.Sequential(
        nn.Linear(9, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(64, 32),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(32, 1),
        )

In [ ]:
EPOCHS = 120
LR = 0.001
WEIGHT_DECAY = 1e-4
N_ENSEMBLE = 5

net = build_model()
opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

In [ ]:
loss_fn2 = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
config = {
    "model": "Ensemble",
    "optimizer": str(opt.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "pos_weight": pos_weight,
    "loss": str(loss_fn2.__class__.__name__),
    "n_ensemble": N_ENSEMBLE,
    "architecture": str(net)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="ensemble", config=config)

In [ ]:
test_probs = []
logging.info("Запустили обучение моделей ансамбля")

np.random.seed(42)
seeds = np.random.randint(1, 525252, size=N_ENSEMBLE)
ensemble = []
for i in range(N_ENSEMBLE):
    seed = seeds[i]
    torch.manual_seed(seed)
    net = build_model()
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    logging.info(f"Запустили обучение модели {i+1}. Сид: {seed}")


    net.train()
    for epoch in range(1, EPOCHS + 1):
        epoch_loss = 0
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn2(net(xb), yb)
            loss.backward()
            epoch_loss += loss.item()
            opt.step()
        
        logging.info(f"эпоха: {epoch}, loss:  {round(epoch_loss, 4)}")
        wandb.log({"epoch": epoch, f"ensemble/{i}/train_loss": epoch_loss})
    
    ensemble.append(net.state_dict())

    net.eval()
    with torch.no_grad():
        test_probs.append(torch.sigmoid(net(X_test)).numpy())

    logging.info(f"Сеть {i} обучена")
    print("сеть", i, "обучена")

probs2 = np.mean(test_probs, axis=0)

In [ ]:
auc_score = roc_auc_score(y_true, probs2)
logging.info(f"На тесте ROC_AUC: {auc_score}")
auc_score

In [ ]:
pre_score = average_precision_score(y_true, probs2)
logging.info(f"На тесте Precision: {pre_score}")
pre_score

In [ ]:
results = wandb.Table(columns=['model', 'test/avg_roc_auc', 'test/avg_precision'])
results.add_data("ensemble", auc_score, pre_score)
wandb.log({"results": results})

In [ ]:
import pickle

pickle.dump(ensemble, open("models/model_fraud_ensemble.pkl", 'wb'))
logging.info("Сохранили веса моделей ансамбля в папку models")

In [ ]:
artifact = wandb.Artifact(name="model_fraud_ensemble.pkl", type="model", description="Ансамбль моделей для определения Фрода")

artifact.add_file("models/model_fraud_ensemble.pkl")
artifact.add_file(log_file)
run.log_artifact(artifact)


In [ ]:
run.finish()

In [36]:
wandb.finish()
